<a href="https://colab.research.google.com/github/chalini99/Capstone-Project-files/blob/main/Model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [2]:
!git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git

Cloning into 'PlantVillage-Dataset'...
remote: Enumerating objects: 163219, done.
remote: Counting objects: 100% (163219/163219), done.
remote: Compressing objects: 100% (163125/163125), done.
remote: Total 163219 (delta 93), reused 163215 (delta 93), pack-reused 0 (from 0)
Receiving objects: 100% (163219/163219), 2.00 GiB | 33.74 MiB/s, done.
Resolving deltas: 100% (93/93), done.
Updating files: 100% (182404/182404), done.


In [3]:
import os

DATASET_DIR = "/content/PlantVillage-Dataset/raw/color"

print("Dataset exists:", os.path.exists(DATASET_DIR))
print("Number of classes:", len(os.listdir(DATASET_DIR)))

Dataset exists: True
Number of classes: 38


In [4]:
from google.colab import files

uploaded = files.upload()

Saving test.csv to test.csv
Saving train.csv to train.csv
Saving val.csv to val.csv


In [5]:
import pandas as pd

train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/val.csv")
test_df = pd.read_csv("/content/test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (43444, 4)
Validation: (5430, 4)
Test: (5431, 4)


In [6]:
!pip install -q timm

In [7]:
print(train_df["image_path"].iloc[0])

C:\PlantVillage-Dataset\raw\color\Tomato___Bacterial_spot\526959cc-f05a-46fb-8625-ed60fdac30cd___GCREC_Bact.Sp 3032.JPG


In [8]:
def fix_path(path):
    path = path.replace(
        r"C:\PlantVillage-Dataset\raw\color",
        "/content/PlantVillage-Dataset/raw/color"
    )
    return path.replace("\\", "/")

train_df["image_path"] = train_df["image_path"].apply(fix_path)
val_df["image_path"] = val_df["image_path"].apply(fix_path)
test_df["image_path"] = test_df["image_path"].apply(fix_path)

print(train_df["image_path"].iloc[0])

/content/PlantVillage-Dataset/raw/color/Tomato___Bacterial_spot/526959cc-f05a-46fb-8625-ed60fdac30cd___GCREC_Bact.Sp 3032.JPG


In [9]:
from PIL import Image
import os

image_path = train_df["image_path"].iloc[0]

print("File exists:", os.path.exists(image_path))

image = Image.open(image_path)
print("Image size:", image.size)
print("Image mode:", image.mode)

File exists: True
Image size: (256, 256)
Image mode: RGB


In [10]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image


class PlantDiseaseDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image = Image.open(row["image_path"]).convert("RGB")
        label = int(row["label"])

        if self.transform:
            image = self.transform(image)

        return image, label

In [11]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transforms created successfully!")

Transforms created successfully!


In [12]:
train_dataset = PlantDiseaseDataset(
    train_df,
    transform=train_transform
)

val_dataset = PlantDiseaseDataset(
    val_df,
    transform=val_test_transform
)

test_dataset = PlantDiseaseDataset(
    test_df,
    transform=val_test_transform
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 43444
Validation dataset: 5430
Test dataset: 5431


In [13]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders created successfully!")

DataLoaders created successfully!


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

print("Device:", device)
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

Device: cuda
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])


In [15]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

weights = EfficientNet_B0_Weights.DEFAULT

model = efficientnet_b0(weights=weights)

num_classes = 38

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    num_classes
)

model = model.to(device)

print(model.classifier)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 88.5MB/s]


Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=38, bias=True)
)


In [16]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print("Loss function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)

Loss function: CrossEntropyLoss()
Optimizer: AdamW


In [17]:
model.eval()

images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)
print("Device:", outputs.device)


Input shape: torch.Size([32, 3, 224, 224])
Output shape: torch.Size([32, 38])
Device: cuda:0


In [18]:
import time

EPOCHS = 10
best_val_accuracy = 0.0

train_history = []
val_history = []

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print("-" * 40)

    # Training
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    start_time = time.time()

    for images, labels in train_loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_accuracy = correct / total

    # Validation
    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_running_loss / val_total
    val_accuracy = val_correct / val_total

    epoch_time = time.time() - start_time

    train_history.append({
        "loss": train_loss,
        "accuracy": train_accuracy
    })

    val_history.append({
        "loss": val_loss,
        "accuracy": val_accuracy
    })

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_accuracy:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val Accuracy: {val_accuracy:.4f}")
    print(f"Time: {epoch_time / 60:.2f} minutes")

    # Save best model
    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            "/content/efficientnet_b0_best.pth"
        )

        print("✅ Best model saved!")

print("\nTraining complete!")
print("Best validation accuracy:", best_val_accuracy)


Epoch 1/10
----------------------------------------
Train Loss: 0.3925
Train Accuracy: 0.9130
Val Loss: 0.0284
Val Accuracy: 0.9910
Time: 3.93 minutes
✅ Best model saved!

Epoch 2/10
----------------------------------------
Train Loss: 0.0429
Train Accuracy: 0.9879
Val Loss: 0.0250
Val Accuracy: 0.9912
Time: 3.94 minutes
✅ Best model saved!

Epoch 3/10
----------------------------------------
Train Loss: 0.0258
Train Accuracy: 0.9923
Val Loss: 0.0087
Val Accuracy: 0.9972
Time: 3.94 minutes
✅ Best model saved!

Epoch 4/10
----------------------------------------
Train Loss: 0.0194
Train Accuracy: 0.9937
Val Loss: 0.0118
Val Accuracy: 0.9963
Time: 3.95 minutes

Epoch 5/10
----------------------------------------
Train Loss: 0.0167
Train Accuracy: 0.9944
Val Loss: 0.0090
Val Accuracy: 0.9967
Time: 3.95 minutes

Epoch 6/10
----------------------------------------
Train Loss: 0.0132
Train Accuracy: 0.9962
Val Loss: 0.0108
Val Accuracy: 0.9971
Time: 3.95 minutes

Epoch 7/10
----------------